# 02 — Single-Indicator IC Analysis

**Phase 2 deliverable** from `docs/research_plan.md`.

Goal: for each candidate volume/price indicator, measure predictive power against forward 15-min, 1-hour, 1-day, and 4-day returns on SPY/QQQ/IWM. Reject indicators where `|IC| < 0.02` (no edge) per the plan's threshold. The survivors feed into Phase 3 strategy construction.

Indicators tested:
- `vwap_deviation` — close vs intraday VWAP (% deviation)
- `obv_zscore` — OBV normalized over 1-day rolling window
- `adl_zscore` — Accumulation/Distribution Line, normalized
- `mfi` — Money Flow Index, bounded [0, 100]
- `cmf` — Chaikin Money Flow, bounded [-1, 1]
- `volume_zscore`, `volume_ratio` — raw volume features

Implementation: `indicators.py` (copied here per Option A; canonical at `../analysis/indicators.py`).

In [ ]:
# region imports
from AlgorithmImports import *
from QuantConnect.Research import QuantBook

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime, timedelta

from indicators import compute_all, ALL as INDICATOR_FNS
from ic_calculator import compute_ic, quantile_test, ic_verdict
# endregion

qb = QuantBook()
pd.set_option('display.float_format', '{:,.4f}'.format)
pd.set_option('display.width', 140)

TICKERS = ['SPY', 'QQQ', 'IWM']
PERIODS = (1, 4, 24, 96)        # 15m, 1h, 1d, 4d at 15-min bars
END   = datetime(2024, 12, 31)
START = END - timedelta(days=365 * 2)   # 2 years

## 1. Fetch and resample to 15-min

Single batch fetch, resample per-symbol. Same pipeline as `01_data_exploration.ipynb` but extended to 2 years for more IC statistical power.

In [ ]:
symbols = {t: qb.add_equity(t, Resolution.MINUTE).symbol for t in TICKERS}
history = qb.history(list(symbols.values()), START, END, Resolution.MINUTE)
print(f'minute rows: {len(history):,}')

def resample_15min(df: pd.DataFrame) -> pd.DataFrame:
    return df.resample('15min').agg({
        'open':   'first',
        'high':   'max',
        'low':    'min',
        'close':  'last',
        'volume': 'sum',
    }).dropna(subset=['open'])

bars_15m = {t: resample_15min(history.loc[sym]) for t, sym in symbols.items()}
for t, df in bars_15m.items():
    print(f'  {t}: {len(df):,} 15-min bars  ({df.index.min().date()} → {df.index.max().date()})')

## 2. Compute indicators on every symbol

In [ ]:
indicators_by_sym = {t: compute_all(df) for t, df in bars_15m.items()}
returns_by_sym   = {t: df['close'].pct_change() for t, df in bars_15m.items()}

# preview SPY
print('Indicators on SPY (last 5 rows, non-null):')
indicators_by_sym['SPY'].dropna().tail()

## 3. IC at multiple horizons, per symbol

Spearman rank IC. Rows = indicator, columns = forward horizon in 15-min bars. `1` ≈ 15 min, `4` ≈ 1 h, `24` ≈ 1 trading day, `96` ≈ 4 trading days.

Forward returns cross day-boundaries (overnight gaps included). That adds noise to multi-bar horizons but doesn't bias the IC direction; we accept it for Phase 2.

In [ ]:
records = []
for sym, ind_df in indicators_by_sym.items():
    rets = returns_by_sym[sym]
    for name in ind_df.columns:
        for period, r in compute_ic(ind_df[name], rets, periods=PERIODS).items():
            records.append({
                'symbol':    sym,
                'indicator': name,
                'period':    period,
                'ic':        r.ic,
                'pvalue':    r.pvalue,
                'n':         r.n,
            })
ic_long = pd.DataFrame(records)

for sym in TICKERS:
    table = (ic_long[ic_long.symbol == sym]
             .pivot_table(values='ic', index='indicator', columns='period')
             .reindex(INDICATOR_FNS.keys()))
    print(f'\n── {sym} ── IC by horizon (rows=indicator, cols=bars ahead) ──')
    print(table.to_string())

## 4. Averaged across symbols + verdict

Mean IC across SPY/QQQ/IWM gives a robustness check — an indicator that only works on one ticker is less interesting than one that works on all three. Verdict applies the plan's `|IC|` thresholds (reject < 0.02; marginal < 0.05; good < 0.10; strong above).

In [ ]:
avg_ic = (ic_long.groupby(['indicator', 'period'])['ic']
          .mean().unstack('period').reindex(INDICATOR_FNS.keys()))
print('Mean IC across SPY/QQQ/IWM (rows=indicator, cols=bars ahead):')
print(avg_ic.to_string())
print()

# Per indicator, the horizon with the largest |IC| and its signed value
best_period = avg_ic.abs().idxmax(axis=1)
best_ic     = pd.Series({name: avg_ic.loc[name, best_period[name]] for name in avg_ic.index})
best = pd.DataFrame({
    'best_period': best_period,
    'best_ic':     best_ic,
    'verdict':     best_ic.apply(ic_verdict),
})
print('Best-horizon summary:')
print(best.to_string())

## 5. Quantile test for indicators that survive

Survivors (verdict ≠ `reject`) get the monotonicity check: bucket the indicator into 5 quantiles per symbol, plot mean forward 1-day return per bucket. A real factor produces a monotonic ramp; a fluke produces flat or U-shaped buckets.

In [ ]:
survivors = best[best.verdict != 'reject'].index.tolist()
print(f'survivors: {survivors}')

for name in survivors:
    print(f'\n── {name} — quantile test (period=24, ≈ 1 day) ──')
    for sym in TICKERS:
        q = quantile_test(indicators_by_sym[sym][name], returns_by_sym[sym],
                          n_quantiles=5, period=24)
        print(f'  {sym}:  ' + '  '.join(
            f'Q{i}={q.loc[i,"mean"]:+.5f}' for i in q.index
        ))

## 6. Rolling IC stability (top survivors only)

An indicator whose IC sign flips across the sample is not actionable. Plot 1-month rolling IC for each survivor on SPY.

At 15-min bars, 1 month ≈ 96 × 21 = 2016 bars.

In [ ]:
from ic_calculator import rolling_ic

if survivors:
    fig, axes = plt.subplots(len(survivors), 1, figsize=(11, 2.5 * len(survivors)), squeeze=False)
    rets_spy = returns_by_sym['SPY']
    for ax, name in zip(axes[:, 0], survivors):
        ric = rolling_ic(indicators_by_sym['SPY'][name], rets_spy, window=2016, period=24)
        ax.plot(ric.index, ric.values, linewidth=0.9)
        ax.axhline(0, color='red', alpha=0.5, linewidth=0.8)
        ax.set_title(f'SPY rolling 1-month IC — {name} (period=24)', fontsize=10)
        ax.set_ylim(-0.3, 0.3)
    plt.tight_layout()
else:
    print('No survivors — skipping rolling IC plot.')

## 7. Verdict and next steps

Three questions to answer before moving to Phase 3:

1. **Which indicators survived `|IC| < 0.02`?** → look at the `verdict` column in section 4.
2. **Did their quantile means rise/fall monotonically?** → section 5.
3. **Was the rolling IC sign stable?** → section 6.

Indicators passing all three are candidates for Phase 3 strategy construction. The sign of `best_ic` tells the direction (positive → indicator high implies higher forward return; negative → indicator high implies lower forward return).

In [ ]:
# Final Phase 2 summary, copy to reports/ when done.
summary = best.copy()
summary['direction'] = np.where(summary['best_ic'] > 0, 'positive', 'negative')
print('Phase 2 summary table:')
print(summary.to_string())